In [2]:
import __init__

Navigated to package root


#### 1st pass finetune - Mattergen XRD
- Dataset Source: [Mattergen Alex-MP-20](https://github.com/microsoft/mattergen/tree/main/data-release/alex-mp)
  - Columns: Database (manual) 
  - Reduced Formula (Source)
  - CIF (pmg - Cifwriter with symprec 0.1)
  - XRD 'Condition Vector' (with [_calculate_theor_XRD.py](_utils/_preprocessing/_calculate_theor_XRD.py))
    - pmg - XRDCalculator(wavelength="CuKa")
    - top 20 most intense peaks selected ($2\theta$ and int)
    - Normalisations
      - $2\theta$ min-max for 0,90
      - intensities min-max for 0,100
- Deduplicated
- Cleaned for CIF augmentation
  -  Note: I didnt filter to context length here because it was not implemented yet, but filter to context was flagged as True during model training which effectively does the same thing (less efficient)
- dataset pushed to HuggingFace as: c-bone/mattergen_XRD (90:10 train/valid sets)

In [ ]:
!torchrun --nproc_per_node=2 _train.py --config '_config_files/training/conditional/xrd_studies/mattergen_XRD-slider.jsonc'

CHILLI

### Training
> **Note**: Here the hyperparamters change compared to regular finetuning because its 2nd pass. Backbone learning rates were set to decay from $5\times10^{-8}$ to $5\times10^{-10}$, while the learning rates for the newly initialised conditioning parameters were set 100 times higher

In [2]:
import __init__
import pandas as pd

df = pd.read_parquet('/home/cyprien/Data_Gen/chili100k_data.parquet')
# drop the 'Split' column
df = df.drop(columns=['Split'])
df.to_parquet('/home/cyprien/Data_Gen/chili100k_data.parquet', index=False)

In [4]:
!python _utils/_preprocessing/_deduplicate.py \
  --input_parquet /home/cyprien/Data_Gen/chili100k_data.parquet \
  --output_parquet HF-databases/chili100k/chili100k_dedup.parquet

Loading data from /home/cyprien/Data_Gen/chili100k_data.parquet...

Amount of entries before filtering: 20,882
Filtered down to: 20,882

Starting deduplication...
Deduplicating table CIF data...
Processing CIF entries: 100%|██████████| 20882/20882 [00:00<00:00, 29046.40it/s]
Number of entries after deduplication: 14,118

Saving deduplicated data to HF-databases/chili100k/chili100k_dedup.parquet...
Process completed successfully.


In [ ]:
df = pd.read_parquet('HF-databases/chili100k/chili100k_dedup.parquet')


# emove any cif wih partial occapncy with something like this:
# if not task_dict['include_occupancy_structures']:
#     for site in structure:
#         occ = list(site.species.as_dict().values())[0]
#         if occ < 1:
#             raise Exception("Occupancy below 1.0 found")



,Material ID,Database,Reduced Formula,CIF
0,1532947,COD,Rb2TbF6,# generated using pymatgen\ndata_Rb2TbF6\n_sym...
1,9008377,COD,Mn4Be3Si3SO12,# generated using pymatgen\ndata_Mn4Be3Si3SO12...
2,1530788,COD,CeNi5Sn,# generated using pymatgen\ndata_CeNi5Sn\n_sym...
3,1007080,COD,KMg(PO3)3,# generated using pymatgen\ndata_KMg(PO3)3\n_s...
4,9003401,COD,LiAl(SiO3)2,# generated using pymatgen\ndata_LiAl(SiO3)2\n...
...,...,...,...,...
14113,1011087,COD,FeHO2,# generated using pymatgen\ndata_FeHO2\n_symme...
14114,1523128,COD,Li3Pb,# generated using pymatgen\ndata_Li3Pb\n_symme...
14115,9008060,COD,Hg(NCl)2,# generated using pymatgen\ndata_Hg(NCl)2\n_sy...
14116,9001514,COD,Al2SiO6,# generated using pymatgen\ndata_Al2SiO6\n_sym...


In [ ]:
!python _utils/_preprocessing/_calculate_theor_XRD.py \
    --input_parquet HF-databases/chili100k/chili100k_dedup.parquet \
    --output_parquet HF-databases/chili100k/chili100k_dedup.parquet \
    --num_workers 16

Loading database from HF-databases/chili100k/chili100k_dedup.parquet
Loaded 14118 entries
Processing XRD patterns for the entries
Generating XRD patterns: 100%|███| 14118/14118 [01:26<00:00, 163.89structures/s]
Computing condition vectors from XRD patterns
Computing condition vectors: 100%|█| 14118/14118 [00:01<00:00, 14079.65structure
Theta range: 0-90, Intensity range: 0-100
Saving results to F-databases/chili100k/chili100k_dedup.parquet
done


In [ ]:
# python _utils/_metrics/VUN_metrics.py \
#   --input_parquet generated_structures_processed.parquet \
#   --huggingface_dataset "c-bone/mp_20" \
#   --output_parquet vun_results.parquet \
#   --num_workers 8

!python _utils/_metrics/VUN_metrics.py \
    --input_parquet HF-databases/chili100k/chili100k_dedup.parquet \
    --huggingface_dataset "c-bone/lematerial_clean" \
    --load_processed_data "HF-databases/lematerial/lematerial_dedup.parquet" \
    --output_parquet HF-databases/chili100k/chili100k_dedup_vun.parquet \
    --output_csv HF-databases/chili100k/chili100k_dedup_vun.csv \
    --check_comp_novelty \
    --num_workers 32 \
    --bond_length_acceptability_cutoff 0.0

In [ ]:
!python _utils/_preprocessing/_cleaning.py \
    --input_parquet HF-databases/chili100k/chili100k_dedup_vun.parquet \
    --output_parquet HF-databases/chili100k/chili100k_count.parquet \
    --num_workers 32 \
    --count_tokens

In [ ]:
df_count = pd.read_parquet('HF-databases/chili100k/chili100k_count.parquet')
df_dedup = pd.read_parquet('HF-databases/chili100k/chili100k_dedup_vun.parquet')
# remove rows in df_dedup where token_count (from df_count) is above 1536, where common column with id is 'Material ID'
df_merged = pd.merge(df_dedup, df_count[['Material ID', 'token_count']], on='Material ID', how='inner')
df_filtered = df_merged[df_merged['token_count'] <= 1536]
# save the filtered dataframe to a new parquet file
df_filtered.to_parquet('HF-databases/chili100k/chili100k_dedup_vun_filtered.parquet', index=False)

14091


,Material ID,Database,Reduced Formula,Generated CIF,condition_vector,is_valid,is_unique,is_novel,is_comp_novel,token_count
0,1532947,COD,Rb2TbF6,# generated using pymatgen\ndata_Rb2TbF6\n_sym...,"[0.29,0.29,0.326,0.32,0.323,0.166,0.512,0.438,...",True,True,False,False,431
1,9008377,COD,Mn4Be3Si3SO12,# generated using pymatgen\ndata_Mn4Be3Si3SO12...,"[0.296,0.523,0.523,0.458,0.385,0.973,0.973,0.9...",True,True,False,False,495
2,1530788,COD,CeNi5Sn,# generated using pymatgen\ndata_CeNi5Sn\n_sym...,"[0.407,0.457,0.383,0.513,0.476,0.426,0.866,0.5...",True,True,False,False,495
3,1007080,COD,KMg(PO3)3,# generated using pymatgen\ndata_KMg(PO3)3\n_s...,"[0.266,0.364,0.317,0.347,0.202,0.476,0.632,0.5...",True,True,False,False,465
4,9003401,COD,LiAl(SiO3)2,# generated using pymatgen\ndata_LiAl(SiO3)2\n...,"[0.336,0.358,0.228,0.432,0.314,0.299,0.53,0.64...",True,True,False,False,494
...,...,...,...,...,...,...,...,...,...,...
14106,1011087,COD,FeHO2,# generated using pymatgen\ndata_FeHO2\n_symme...,"[0.235,0.406,0.368,0.589,0.385,0.653,0.599,0.4...",True,True,True,False,397
14107,1523128,COD,Li3Pb,# generated using pymatgen\ndata_Li3Pb\n_symme...,"[0.256,0.5,0.296,0.423,0.67,0.69,0.764,0.956,0...",True,True,True,False,338
14108,9008060,COD,Hg(NCl)2,# generated using pymatgen\ndata_Hg(NCl)2\n_sy...,"[0.171,0.243,0.243,0.299,0.346,0.346,0.462,0.4...",True,True,True,False,367
14109,9001514,COD,Al2SiO6,# generated using pymatgen\ndata_Al2SiO6\n_sym...,"[0.335,0.305,0.264,0.474,0.422,0.76,0.722,0.48...",True,True,False,False,461


In [11]:
import __init__
import pandas as pd
# load df
df = pd.read_parquet('HF-databases/chili100k/chili100k_dedup_vun_filtered.parquet')

df = df[~((df['is_valid'] == False) & (df['is_unique'] == False))]
# remove those columns
df = df.drop(columns=['is_valid', 'is_unique'])

# Define strictly mutually exclusive masks to prevent data overlap
# mask_comp_novel = df['is_comp_novel'] == True and 'Reduced Formula' is unique in the dataset
# We want to ensure that if a composition is marked as novel, it does not have duplicates in the dataset, which could leak information across splits. Therefore, we check for both conditions: is_comp_novel is True and the Reduced Formula is not duplicated (i.e., unique).
mask_comp_novel = (df['is_comp_novel'] == True) & (df['Reduced Formula'].duplicated(keep=False) == False)
print(f"Number of comp novel structures: {mask_comp_novel.sum()}")
mask_struct_novel = (df['is_novel'] == True) & (df['is_comp_novel'] == False)
mask_in_dist = df['is_novel'] == False 

# Sample the test set
test_comp = df[mask_comp_novel].sample(n=500, random_state=1)
test_struct = df[mask_struct_novel].sample(n=500, random_state=1)
test_in_dist = df[mask_in_dist].sample(n=500, random_state=1)
test_indices = pd.concat([test_comp, test_struct, test_in_dist]).index

# Filter out the test set to define the remaining pool
remaining_df = df.drop(test_indices)

# Sample the validation set (1500 from in-distribution structures)
val_indices = remaining_df[remaining_df['is_novel'] == False].sample(n=1500, random_state=1).index

df['Split'] = 'train'
df.loc[val_indices, 'Split'] = 'val'
df.loc[test_indices, 'Split'] = 'test'

print(df['Split'].value_counts())

# renam 'Generated CIF' to CIF
df = df.rename(columns={'Generated CIF': 'CIF'})

df.to_parquet('HF-databases/chili100k/chili100k_dedup_leakproof.parquet', index=False)


Number of comp novel structures: 1589
Split
train    11091
val       1500
test      1500
Name: count, dtype: int64


In [12]:
!python _utils/_preprocessing/_cleaning.py \
    --input_parquet 'HF-databases/chili100k/chili100k_dedup_leakproof.parquet' \
    --output_parquet 'HF-databases/chili100k/chili100k_clean.parquet' \
    --num_workers 32 \
    --count_tokens

Loading data from HF-databases/chili100k/chili100k_dedup_leakproof.parquet as Parquet with zstd compression...

Lets augment the CIFs now (parallelizing sometimes takes a min before speeding up
Number of CIFs before preprocessing: 14091
Number of workers: 32
100%|███████████████████████████████████| 14091/14091 [00:04<00:00, 3185.52it/s]
Number of CIFs before filtering out bad ones:  14091
Number of CIFs after filtering: 14091

Counting tokens in CIFs and adding 'token_count' column...
Tokenizer validation passed: token vocabulary is consistent.
Tokenizer validation passed: token vocabulary is consistent.
Tokenizer validation passed: token vocabulary is consistent.
Tokenizer validation passed: token vocabulary is consistent.
Tokenizer validation passed: token vocabulary is consistent.
Tokenizer validation passed: token vocabulary is consistent.
Tokenizer validation passed: token vocabulary is consistent.
Tokenizer validation passed: token vocabulary is consistent.
Tokenizer validation 

In [13]:
import pandas as pd
df = pd.read_parquet('HF-databases/chili100k/chili100k_clean.parquet')
df

,Material ID,Database,Reduced Formula,CIF,condition_vector,is_novel,is_comp_novel,token_count,Split
0,1532947,COD,Rb2TbF6,data_[Rb8Tb4F24]\nloop_\n _atom_type_symbol\n ...,"[0.29,0.29,0.326,0.32,0.323,0.166,0.512,0.438,...",False,False,431,train
1,9008377,COD,Mn4Be3Si3SO12,data_[Mn8Be6Si6S2O24]\nloop_\n _atom_type_symb...,"[0.296,0.523,0.523,0.458,0.385,0.973,0.973,0.9...",False,False,495,train
2,1530788,COD,CeNi5Sn,data_[Ce4Ni20Sn4]\nloop_\n _atom_type_symbol\n...,"[0.407,0.457,0.383,0.513,0.476,0.426,0.866,0.5...",False,False,495,train
3,1007080,COD,KMg(PO3)3,data_[K2Mg2P6O18]\nloop_\n _atom_type_symbol\n...,"[0.266,0.364,0.317,0.347,0.202,0.476,0.632,0.5...",False,False,465,train
4,9003401,COD,LiAl(SiO3)2,data_[Li4Al4Si8O24]\nloop_\n _atom_type_symbol...,"[0.336,0.358,0.228,0.432,0.314,0.299,0.53,0.64...",False,False,494,train
...,...,...,...,...,...,...,...,...,...
14086,1011087,COD,FeHO2,data_[Fe4H4O8]\nloop_\n _atom_type_symbol\n _a...,"[0.235,0.406,0.368,0.589,0.385,0.653,0.599,0.4...",True,False,397,train
14087,1523128,COD,Li3Pb,data_[Li12Pb4]\nloop_\n _atom_type_symbol\n _a...,"[0.256,0.5,0.296,0.423,0.67,0.69,0.764,0.956,0...",True,False,338,train
14088,9008060,COD,Hg(NCl)2,data_[Hg2N4Cl4]\nloop_\n _atom_type_symbol\n _...,"[0.171,0.243,0.243,0.299,0.346,0.346,0.462,0.4...",True,False,367,train
14089,9001514,COD,Al2SiO6,data_[Al8Si4O24]\nloop_\n _atom_type_symbol\n ...,"[0.335,0.305,0.264,0.474,0.422,0.76,0.722,0.48...",False,False,461,test


In [14]:
!python _utils/_preprocessing/_save_dataset_to_HF.py \
    --input_parquet 'HF-databases/chili100k/chili100k_clean.parquet' \
    --output_parquet 'HF-databases/chili100k_strat.parquet' \
    --save_hub

Loading Hugging Face API key from API_keys.jsonc
Loading data from HF-databases/chili100k/chili100k_clean.parquet as Parquet with zstd compression
Splitting dataset according to the 'Split' column
Train columns: ['Material ID', 'Database', 'Reduced Formula', 'CIF', 'condition_vector', 'is_novel', 'is_comp_novel', 'token_count']
README.md: 100%|███████████████████████████████| 777/777 [00:00<00:00, 2.84MB/s]
Dataset saved to Hugging Face Hub as c-bone/chili100k_strat
